# VueCore
- implements at the moment not too many figures
- will be used to define plotting functions for certain analysis types in our
  core analysis library
- at the moment the basic plots in vuecore only support the plotly backend
- more complex analysis plots are available as static and interactive plots:
  in the namespace `vuecore.static` and `vuecore.interactive`


Colab specific setup:

In [ ]:
import os

IN_COLAB = "COLAB_GPU" in os.environ

if IN_COLAB:
    !pip install vuecore

Import libraries

In [ ]:
import os
import pathlib

import pandas as pd
import vuecore
from vuecore.plots.basic.histogram import create_histogram_plot

IN_COLAB = "COLAB_GPU" in os.environ

## Proteomics data example

- load from repository if in colab

In [ ]:
dir_data = pathlib.Path("data")
fname = dir_data / "proteins" / "proteins.csv"
if IN_COLAB:
    fname = (
        "https://raw.githubusercontent.com/biosustain/dsp_workshop_dataviz_python"
        "/refs/heads/main/data/proteins/proteins.csv"
    )
df = (
    pd.read_csv(fname, index_col=0)
    .rename_axis("Protein_ID", axis=1)
    .stack()
    .reset_index(name="Intensity")
)
df

In [ ]:
# to be continued
# Generate the advanced histogram plot
fig = create_histogram_plot(
    data=df,
    x="Intensity",
    color="Reference",
    barmode="overlay",
    histnorm="probability density",
    title="Protein intensities by sample",
    subtitle="Histogram with probability density normalized",
    labels={"Intensity": "Protein Intensity", "Reference": "Sample"},
    hover_data=["Protein_ID"],
    opacity=0.75,
)
fig

For now vuecore is built on top of ploltly:

In [ ]:
type(fig)

## Decomposition plots (PCA)

Dimensionality reduction results (PCA, t-SNE, UMAP) from `acore` can be plotted
directly with `vuecore.dimensionality_reduction`, as a static (matplotlib) or an
interactive (plotly) figure. The proteins are the features, the samples are the
observations, so we need the wide format (samples as rows) and a `group` column.

In [ ]:
from acore.exploratory_analysis.decomposition import run_pca
from vuecore.dimensionality_reduction import (
    get_embedding_plot_interactive,
    get_embedding_plot_static,
)

In [ ]:
df_wide = pd.read_csv(fname, index_col=0)
df_wide["group"] = df_wide.index.str.split("_").str[0]  # DMSO or Suf
df_wide[["group"]]

`run_pca` returns the components, loadings and explained variance, and the
axis names (including explained variance). Proteins with missing values are
dropped per default.

In [ ]:
pca_results, pca_annotation = run_pca(df_wide, group="group")
pca_results[0]

Static version (matplotlib figure):

In [ ]:
fig_static = get_embedding_plot_static(
    pca_results,
    pca_annotation,
    group="group",
    title="PCA of protein intensities",
)
fig_static

Interactive version (plotly figure) - hover over the samples. The loadings of
the proteins contributing most are added as arrows (biplot).

In [ ]:
fig_interactive = get_embedding_plot_interactive(
    pca_results,
    pca_annotation,
    group="group",
    title="PCA of protein intensities",
)
fig_interactive